<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎤 Dots.TTS - Zero-Shot Voice Cloning & TTS</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab T4 GPU Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>2B Parameter Continuous Autoregressive Text-to-Speech System with SOAR Alignment & MeanFlow Distillation</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### Features & Optimizations
| Feature | Description |
|---|---|
| ⚡ **CUDA-Graph T4 Engine** | DiT flow steps and Qwen decode replayed as CUDA graphs, one preallocated KV cache (no compile wait, no OOM) |
| 🎙️ **Zero-Shot Voice Cloning** | Clone any speaker identity using 3 to 15 seconds of reference audio |
| 🚀 **MeanFlow 4-Step (Default)** | Fastest model (`dots.tts-mf`): 4 distilled steps with fused guidance; SOAR one click away |
| 📊 **Clean Progress Bar** | One progress bar per request (download, load, generation) in the UI and the notebook, no log spam |
| 🎯 **SOAR Alignment** | Self-Corrective Alignment variant (`dots.tts-soar`) for highest speaker resemblance |
| 📝 **Automatic Transcription** | Built-in OpenAI Whisper (GPU) if reference text is omitted |
| 📜 **Long Text** | Long text is split at sentence ends into ~1-minute parts, generated one by one and joined with natural pauses |
| 🎯 **Natural Pace** | Reference cut at a pause + accurate Whisper transcript; optional pace matching gently slows only the sections that come out rushed |
| 📊 **Live Progress Everywhere** | The in-page progress bar updates on both the Gradio and the Cloudflare link |
| 🌐 **Dual Tunneling** | Rock-solid Cloudflare Quick Tunnel (`*.trycloudflare.com`) + Gradio Public Share |
| 🎛️ **3-Button Control Interface** | Standard AIQUEST UI with Generate Audio, Stop, and Clear controls |

---

### Quick Start
1. Ensure GPU runtime is active: **Runtime -> Change runtime type -> T4 GPU**
2. Run **Step 1** to configure the environment, install system tools, and install dots.tts with an in-place progress tracker.
3. Run **Step 2** to launch the interactive voice cloning studio with both Cloudflare tunnel and Gradio share links!


In [1]:
#@title 📦 Step 1: Environment Setup, Dependencies & System Configuration
# Installs ffmpeg + cloudflared, clones studio-dots-ai/dots.tts, patches pyproject.toml, writes the T4 accelerator module, and installs dependencies with clean in-place progress tracking.
import gc
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request
import warnings

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/torchinductor_cache"
os.environ["TORCH_LOGS"] = "-all"

try:
    from IPython.display import clear_output
    HAS_CLEAR = True
except Exception:
    HAS_CLEAR = False

# 1. Hardware Detection & Tensor Core Setup
import torch
PYTHON_VER = sys.version.split()[0]
TORCH_VER = torch.__version__
GPU_NAME = "None"
VRAM_GB = 0.0
CUDA_VER = "None"

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
    CUDA_VER = torch.version.cuda
    torch.backends.cudnn.benchmark = False

TOTAL_STAGES = 5
STAGE_NAMES = [
    "System & GPU Memory Optimization",
    "System Packages Installation (ffmpeg, rubberband, cloudflared)",
    "Repository, Constraints & T4 Accelerator Setup",
    "Python Dependencies Installation (dots.tts, Gradio)",
    "Runtime & Module Imports Verification",
]

def render_dashboard(current_stage, sub_status="", error_msg=None, env_summary=None):
    if HAS_CLEAR:
        clear_output(wait=True)

    bar_width = 32
    if current_stage > TOTAL_STAGES:
        progress_pct = 100
        filled = bar_width
    else:
        progress_pct = int(((current_stage - 1) / TOTAL_STAGES) * 100)
        filled = int(bar_width * ((current_stage - 1) / TOTAL_STAGES))

    bar = "=" * filled + (">" if filled < bar_width else "") + " " * (bar_width - filled - (1 if filled < bar_width else 0))

    lines = []
    lines.append("=" * 65)
    lines.append("🎤 Dots.TTS - Zero-Shot Voice Cloning & TTS Studio")
    lines.append("📺 Created by: AIQUEST Academy")
    lines.append("🔗 YouTube: @AIQuestAcademy | X: @AIQuestAcademy")
    lines.append("=" * 65)
    lines.append(f"🔧 Python: {PYTHON_VER} | PyTorch: {TORCH_VER}")
    if torch.cuda.is_available():
        lines.append(f"✅ GPU Detected: {GPU_NAME} ({VRAM_GB:.1f} GB VRAM)")
        lines.append(f"✅ CUDA Version: {CUDA_VER} | T4 CUDA-Graph Engine")
    else:
        lines.append("⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")
    lines.append("")
    lines.append(f"Progress: [{bar}] {progress_pct}%")

    for i, name in enumerate(STAGE_NAMES):
        idx = i + 1
        if idx < current_stage or current_stage > TOTAL_STAGES:
            lines.append(f"  ✅ [{idx}/{TOTAL_STAGES}] {name}")
        elif idx == current_stage:
            extra = f" ({sub_status})" if sub_status else "..."
            lines.append(f"  ⏳ [{idx}/{TOTAL_STAGES}] {name}{extra}")
        else:
            lines.append(f"  ⏸️  [{idx}/{TOTAL_STAGES}] {name}")

    if error_msg:
        lines.append("\n" + "=" * 65)
        lines.append("❌ INSTALLATION FAILED:")
        lines.append("=" * 65)
        lines.append(error_msg)
    elif env_summary:
        lines.append("\n🔍 Environment Summary:")
        for k, v in env_summary.items():
            lines.append(f"  • {k}: {v}")
        lines.append("=" * 65)
        lines.append("🎉 Step 1 Complete! You can now proceed to Step 2 to launch the studio.")
        lines.append("=" * 65)

    print("\n".join(lines))
    sys.stdout.flush()

def run_quiet_process(cmd, stage_idx, log_file):
    with open(log_file, "a") as log_f:
        proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, text=True)
        t0 = time.time()
        while proc.poll() is None:
            time.sleep(0.5)
            elapsed = int(time.time() - t0)
            status_text = f"running... {elapsed}s"
            if os.path.exists(log_file):
                try:
                    with open(log_file, "r") as rf:
                        raw_lines = rf.readlines()
                        for rl in reversed(raw_lines):
                            rl_clean = rl.strip()
                            if any(rl_clean.startswith(x) for x in ["Collecting", "Downloading", "Building", "Installing", "Preparing", "Processing"]):
                                status_text = f"{rl_clean[:38]}... ({elapsed}s)"
                                break
                except Exception:
                    pass
            render_dashboard(stage_idx, sub_status=status_text)

        rc = proc.poll()
        if rc != 0:
            err_tail = ""
            if os.path.exists(log_file):
                with open(log_file, "r") as rf:
                    err_tail = "".join(rf.readlines()[-25:])
            render_dashboard(stage_idx, error_msg=err_tail)
            raise RuntimeError(f"Command failed with code {rc}:\n{err_tail}")

# ── Stage 1: System & GPU Memory Optimization ──
render_dashboard(1, "Optimizing allocator & dropping kernel caches...")
os.system("echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1")
os.system("echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1")
gc.collect()
time.sleep(0.5)

# ── Stage 2: System Packages Installation (ffmpeg, cloudflared) ──
render_dashboard(2, "Installing ffmpeg, rubberband and downloading cloudflared...")
os.system("apt-get update -qq > /dev/null 2>&1 && apt-get install -y ffmpeg rubberband-cli -qq > /dev/null 2>&1")

if not os.path.exists("/usr/local/bin/cloudflared"):
    os.system("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
time.sleep(0.5)

# ── Stage 3: Repository Configuration & Constraints Patching ──
render_dashboard(3, "Cloning repository & sanitizing constraints...")
REPO_DIR = "dots.tts"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/studio-dots-ai/dots.tts.git", REPO_DIR], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

# Patch pyproject.toml
toml_path = os.path.join(REPO_DIR, "pyproject.toml")
if os.path.exists(toml_path):
    with open(toml_path, "r") as f:
        toml_content = f.read()

    toml_content = re.sub(r'requires-python\s*=\s*["\'].*?["\']', 'requires-python = ">=3.10"', toml_content)
    toml_content = re.sub(r'["\']numpy>=[0-9\.]+["\']', '"numpy"', toml_content)
    toml_content = re.sub(r'numpy>=[0-9\.]+', 'numpy', toml_content)
    toml_content = re.sub(r'["\']torch>=[0-9\.]+["\']', '"torch"', toml_content)
    toml_content = re.sub(r'["\']torchaudio>=[0-9\.]+["\']', '"torchaudio"', toml_content)
    toml_content = re.sub(r'["\']gradio>=[0-9\.]+,<[0-9\.]+["\']', '"gradio"', toml_content)

    with open(toml_path, "w") as f:
        f.write(toml_content)

# Keep upstream sources pristine: all T4 speed-ups are applied at runtime by dots_tts_t4_accel.py
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--", "src"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# T4 accelerator: CUDA-graph DiT + Qwen decode, one preallocated KV cache, SDPA attention
ACCEL_SOURCE = r'''"""Tesla T4 acceleration for studio-dots-ai/dots.tts, applied at runtime.

Call ``apply()`` once, before ``DotsTtsRuntime.from_pretrained(..., optimize=True,
warmup_on_optimize=False, max_generate_length=<max_bucket>)``. Nothing on disk is
modified, so the upstream checkout can stay pristine.

What it changes
- torch.compile is off everywhere (optionally kept for the LLM decode step only).
- The DiT KV-cache attention uses SDPA (mem-efficient kernel on sm_75), not flex.
- The per-bucket DiT KV caches become views of one preallocated workspace: growing
  from bucket 64 to 256 never reallocates, copies or spikes memory, and tensor
  addresses stay fixed across requests.
- Every NextPatchStep ODE step is replayed as a CUDA graph captured from eager
  kernels (no Inductor, so capture takes milliseconds instead of minutes). All graphs
  share one memory pool and read one set of static inputs. This removes the ~700
  kernel launches per DiT forward that dominate eager decoding on Colab CPUs.
- The Qwen single-token decode step (one call per 160 ms patch) is replayed as a
  CUDA graph too. Its StaticCache workspace is reused across requests, so one capture
  serves every request.
- KV prefill runs at the exact prompt length instead of the 192-patch compile pad.
- Loading builds the model directly on the GPU and streams the checkpoint in one
  tensor at a time, so system RAM never holds the 2B model (upstream peaks around
  14 GB of RAM, more than Colab's 12.7 GB). ``set_load_progress`` reports progress.
"""

from __future__ import annotations

import contextlib
import gc
import os
import weakref
from typing import Any

import torch

from dots_tts.modules.backbone import dit_inference as di
from dots_tts.modules.backbone import inference_utils as iu

BUCKETS = (64, 128, 256, 512)

_SETTINGS: dict[str, Any] = {
    "cuda_graphs": True,
    "compile_llm": False,
    "max_bucket": 256,
    "test_static_path_on_cpu": False,
    "load_progress": None,
}
_ORIG: dict[str, Any] = {}


def apply(
    *,
    max_bucket: int = 256,
    cuda_graphs: bool = True,
    compile_llm: bool = False,
) -> None:
    """Install the patches. Safe to call again to change settings before a model load.

    max_bucket: largest DiT/KV bucket in latent patches (1 patch = 160 ms, prompt
        included). Pass the same value as ``max_generate_length`` to the runtime.
        KV workspace = nfe * cfg_branches * max_bucket * 5 tokens * 72 KiB.
    cuda_graphs: replay NextPatchStep and the Qwen single-token decode step as CUDA
        graphs captured from eager kernels.
    compile_llm: use upstream's torch.compile(mode="reduce-overhead") for the Qwen
        decode step instead of the eager-captured graph (one-time compile).
    """
    if max_bucket not in BUCKETS:
        raise ValueError(f"max_bucket must be one of {BUCKETS}, got {max_bucket}.")
    _SETTINGS.update(
        max_bucket=int(max_bucket),
        cuda_graphs=bool(cuda_graphs),
        compile_llm=bool(compile_llm),
    )

    _configure_backends()
    _patch_compile(compile_llm=compile_llm)
    _set_max_bucket(max_bucket)
    _patch_solver()
    _patch_loading()


def set_load_progress(callback) -> None:
    """``callback(file_name, done_bytes, total_bytes)`` while checkpoint files load."""
    _SETTINGS["load_progress"] = callback


# --------------------------------------------------------------------------- #
# Backends and compile policy
# --------------------------------------------------------------------------- #


def _configure_backends() -> None:
    # Flex attention would need a Triton max-autotune compile per bucket and a new
    # BlockMask per patch; SDPA's mem-efficient kernel handles the bool mask on sm_75.
    os.environ["DOTS_TTS_DELAYED_DIT_BACKEND"] = "sdpa"
    if not torch.cuda.is_available():
        return
    # Flash and cuDNN SDPA need sm_80+, so on T4 these two calls change nothing.
    # They only make the choice explicit. Math stays on as the fallback.
    torch.backends.cuda.enable_flash_sdp(False)
    if hasattr(torch.backends.cuda, "enable_cudnn_sdp"):
        torch.backends.cuda.enable_cudnn_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    # The vocoder sees a new length every request; benchmark mode re-tunes every conv.
    torch.backends.cudnn.benchmark = False


def _patch_compile(*, compile_llm: bool) -> None:
    def _selective_compile(raw):
        if compile_llm and getattr(raw, "__name__", "") == "_static_inputs_embeds":
            return torch.compile(raw, mode="reduce-overhead", fullgraph=True, dynamic=False)
        return raw

    iu._compile = _selective_compile


def _set_max_bucket(max_bucket: int) -> None:
    from dots_tts.models.dots_tts.model import DotsTtsModel

    DotsTtsModel._GENERATE_LENGTH_BUCKETS = tuple(b for b in BUCKETS if b <= max_bucket)


# --------------------------------------------------------------------------- #
# Single KV workspace
# --------------------------------------------------------------------------- #


def _kv_workspace(solver, runner, *, nfe: int, branch_batch: int):
    key = (str(runner.device), runner.dtype, int(nfe), int(branch_batch))
    workspace = getattr(solver, "_t4_kv_workspace", None)
    if workspace is None or workspace[0] != key:
        # Graphs point into the old workspace: drop them before it is freed.
        solver._t4_graphs = {}
        solver._t4_kv_workspace = None
        max_tokens = int(_SETTINGS["max_bucket"]) * int(solver.context.unit_len)
        with torch.inference_mode(False):
            cache_k = torch.zeros(
                (
                    int(nfe),
                    runner.num_layers,
                    int(branch_batch),
                    runner.num_heads,
                    max_tokens,
                    runner.head_dim,
                ),
                device=runner.device,
                dtype=runner.dtype,
            )
            cache_v = torch.zeros_like(cache_k)
        workspace = (key, cache_k, cache_v)
        solver._t4_kv_workspace = workspace
    return workspace[1], workspace[2]


def _ensure_cache(self, state, *, sequence, nfe):
    capacity_patches = self._resolve_bucket_patches(sequence.size(1))
    runner = self._get_cached_runner(
        capacity_patches=capacity_patches,
        device=sequence.device,
        dtype=sequence.dtype,
    )
    branch_batch = self.branch_multiplier * int(sequence.size(0))
    ws_k, ws_v = _kv_workspace(self, runner, nfe=nfe, branch_batch=branch_batch)
    capacity = int(runner.capacity_tokens)
    if capacity > ws_k.size(4):
        return _ORIG["_ensure_cache"](self, state, sequence=sequence, nfe=nfe)

    old = state.cache
    same_layout = (
        isinstance(old, di.DiTKvCache)
        and old.nfe == int(nfe)
        and int(old.cache_k.size(2)) == branch_batch
    )
    if (
        same_layout
        and old.capacity_tokens == capacity
        and old.cache_k.data_ptr() == ws_k.data_ptr()
    ):
        return runner, old

    valid_tokens = 0
    if same_layout:
        valid_tokens = min(int(old.valid_tokens), capacity)
        if old.cache_k.data_ptr() != ws_k.data_ptr() and valid_tokens > 0:
            ws_k[..., :valid_tokens, :].copy_(old.cache_k[..., :valid_tokens, :])
            ws_v[..., :valid_tokens, :].copy_(old.cache_v[..., :valid_tokens, :])
    else:
        # New generation: match upstream's zero-initialised cache.
        ws_k.zero_()
        ws_v.zero_()

    cache = di.DiTKvCache(
        capacity_patches=runner.capacity_patches,
        capacity_tokens=capacity,
        nfe=int(nfe),
        cache_k=ws_k[..., :capacity, :],
        cache_v=ws_v[..., :capacity, :],
        valid_tokens=valid_tokens,
    )
    cache._t4_workspace = True
    state.cache = cache
    return runner, cache


# --------------------------------------------------------------------------- #
# CUDA graphs for NextPatchStep
# --------------------------------------------------------------------------- #


# Every live graph, across models. A new graph joins the memory pool of a graph that
# is still alive; a pool whose graphs were all freed (for example after switching
# models) is released by PyTorch, so its handle must never be reused.
_LIVE_GRAPHS: "weakref.WeakSet[torch.cuda.CUDAGraph]" = weakref.WeakSet()


def _capture_graph(fn):
    """Capture ``fn()`` into a new CUDA graph and return ``(graph, outputs)``.

    Graphs replay one at a time and keep their outputs alive, so their temporaries
    can safely share one pool.
    """
    pool = next((graph.pool() for graph in list(_LIVE_GRAPHS)), None)
    graph = torch.cuda.CUDAGraph()
    try:
        with _autocast_without_cache(), torch.cuda.graph(graph, pool=pool):
            outputs = fn()
    except Exception:
        _reset_capture_state()
        raise
    _LIVE_GRAPHS.add(graph)
    return graph, outputs


def _reset_capture_state() -> None:
    """A capture that fails half-way can leave the CUDA RNG convinced a capture is
    still running, which then breaks every later torch.randn. An empty capture in a
    fresh pool runs the RNG's capture prologue and epilogue again and clears it."""
    try:
        torch.cuda.synchronize()
        with torch.cuda.graph(torch.cuda.CUDAGraph()):
            pass
    except Exception:
        pass


def _autocast_without_cache():
    # Graph capture must not populate autocast's weight-cast cache.
    try:
        enabled = torch.is_autocast_enabled("cuda")
    except TypeError:
        enabled = torch.is_autocast_enabled()
    if not enabled:
        return contextlib.nullcontext()
    if hasattr(torch, "get_autocast_dtype"):
        dtype = torch.get_autocast_dtype("cuda")
    else:
        dtype = torch.get_autocast_gpu_dtype()
    return torch.autocast("cuda", dtype=dtype, cache_enabled=False)


class _NextStepGraphs:
    """One CUDA graph per ODE index for a single KV bucket.

    All graphs read the same static input buffers and share one memory pool. They
    are replayed strictly one after another, and each graph's outputs stay alive
    here, so sharing the pool is safe (see ``_capture_graph``).
    """

    def __init__(self, step, *, cache_k, cache_v, example: dict[str, torch.Tensor]):
        self.step = step
        self.cache_k = cache_k
        self.cache_v = cache_v
        with torch.inference_mode(False):  # plain tensors: usable in and out of inference mode
            self.inputs = {name: tensor.clone() for name, tensor in example.items()}
        self.graphs: dict[int, torch.cuda.CUDAGraph] = {}
        self.outputs: dict[int, tuple[torch.Tensor, ...]] = {}

    def load(self, **tensors: torch.Tensor) -> None:
        for name, tensor in tensors.items():
            self.inputs[name].copy_(tensor)

    def _run(self, ode_idx: int):
        kwargs = dict(self.inputs)
        z = kwargs.pop("z")
        all_mods = kwargs.pop("all_mods")
        return self.step(
            z,
            all_mods=all_mods[ode_idx],
            cache_k=self.cache_k[ode_idx],
            cache_v=self.cache_v[ode_idx],
            **kwargs,
        )

    def capture(self, nfe: int) -> None:
        """Capture any missing ODE-step graphs. Nothing is written to the KV cache."""
        if not self.inputs["z"].is_cuda:
            return
        for ode_idx in range(nfe):
            if ode_idx in self.graphs:
                continue
            side = torch.cuda.Stream()
            side.wait_stream(torch.cuda.current_stream())
            with torch.cuda.stream(side):
                self._run(ode_idx)  # warm-up: lazy cuBLAS/SDPA init outside capture
            torch.cuda.current_stream().wait_stream(side)
            graph, outputs = _capture_graph(lambda: self._run(ode_idx))
            self.graphs[ode_idx] = graph
            self.outputs[ode_idx] = outputs

    def __call__(self, ode_idx: int, z: torch.Tensor):
        self.inputs["z"].copy_(z)
        if not z.is_cuda:
            return self._run(ode_idx)
        self.graphs[ode_idx].replay()
        return self.outputs[ode_idx]


def _graphs_for(solver, runner, kv_cache, *, example: dict[str, torch.Tensor]):
    if getattr(solver, "_t4_graphs", None) is None:
        solver._t4_graphs = {}
    key = (
        int(runner.capacity_tokens),
        int(kv_cache.nfe),
        kv_cache.cache_k.data_ptr(),
        tuple(sorted(example)),
        tuple(example["z"].shape),
        example["z"].dtype,
    )
    graphs = solver._t4_graphs.get(key)
    if graphs is None:
        graphs = _NextStepGraphs(
            runner.step,
            cache_k=kv_cache.cache_k,
            cache_v=kv_cache.cache_v,
            example=example,
        )
        solver._t4_graphs[key] = graphs
    return graphs


def _decode_with_kv_cache(
    self,
    z,
    *,
    state,
    sequence,
    cfg_sequence,
    fm_seq_len,
    prefix_len,
    current_hidden,
    cfg_current_hidden,
    all_mods_by_ode,
    schedule,
    nfe,
    batch_size,
    guidance,
    flow_dt,
):
    use_static_path = _SETTINGS["cuda_graphs"] and (
        sequence.is_cuda or _SETTINGS["test_static_path_on_cpu"]
    )
    if use_static_path and not getattr(self, "_t4_graph_failed", False):
        try:
            return _decode_static(
                self,
                z,
                state=state,
                sequence=sequence,
                cfg_sequence=cfg_sequence,
                fm_seq_len=fm_seq_len,
                prefix_len=prefix_len,
                current_hidden=current_hidden,
                cfg_current_hidden=cfg_current_hidden,
                all_mods_by_ode=all_mods_by_ode,
                schedule=schedule,
                nfe=nfe,
                batch_size=batch_size,
                guidance=guidance,
                flow_dt=flow_dt,
            )
        except _FallBack:
            pass
    return _ORIG["_decode_with_kv_cache"](
        self,
        z,
        state=state,
        sequence=sequence,
        cfg_sequence=cfg_sequence,
        fm_seq_len=fm_seq_len,
        prefix_len=prefix_len,
        current_hidden=current_hidden,
        cfg_current_hidden=cfg_current_hidden,
        all_mods_by_ode=all_mods_by_ode,
        schedule=schedule,
        nfe=nfe,
        batch_size=batch_size,
        guidance=guidance,
        flow_dt=flow_dt,
    )


class _FallBack(Exception):
    pass


def _decode_static(
    self,
    z,
    *,
    state,
    sequence,
    cfg_sequence,
    fm_seq_len,
    prefix_len,
    current_hidden,
    cfg_current_hidden,
    all_mods_by_ode,
    schedule,
    nfe,
    batch_size,
    guidance,
    flow_dt,
):
    unit_len = int(self.context.unit_len)
    persistent_len = prefix_len - unit_len
    runner, kv_cache = self._ensure_cache(
        state, sequence=sequence[:, :fm_seq_len], nfe=nfe
    )
    if runner.attn_backend != "sdpa" or not getattr(kv_cache, "_t4_workspace", False):
        raise _FallBack
    if kv_cache.valid_tokens != persistent_len:
        runner.prefill(
            kv_cache=kv_cache,
            prefix_sequence=sequence[:, :persistent_len],
            cfg_prefix_sequence=(
                None if cfg_sequence is None else cfg_sequence[:, :persistent_len]
            ),
            all_mods_by_ode=all_mods_by_ode,
        )
    _block_mask, sdpa_mask = runner.masks_for(valid_persistent_tokens=persistent_len)
    rotary_cos, rotary_sin = runner.rotary_for(start_pos=persistent_len)

    inputs = {
        "prev_unit": sequence[:, persistent_len:prefix_len],
        "current_hidden": current_hidden,
        "all_mods": all_mods_by_ode,
        "sdpa_mask": sdpa_mask,
        "rotary_cos": rotary_cos,
        "rotary_sin": rotary_sin,
    }
    if cfg_sequence is not None:
        inputs.update(
            cfg_prev_unit=cfg_sequence[:, persistent_len:prefix_len],
            cfg_current_hidden=cfg_current_hidden,
            guidance_scale=guidance,
        )
    graphs = _graphs_for(self, runner, kv_cache, example={"z": z, **inputs})
    graphs.load(z=z, **inputs)
    try:
        graphs.capture(nfe)
    except Exception as exc:  # capture failed before any cache write: go eager
        self._t4_graph_failed = True
        self._t4_graphs = {}
        print(f"⚠️ DiT CUDA graph capture failed, using eager steps: {exc}")
        raise _FallBack from exc

    dst = slice(persistent_len, prefix_len)
    for ode_idx in range(nfe):
        vt, new_k, new_v = graphs(ode_idx, z)
        z = schedule.advance(
            z,
            vt,
            ode_idx=ode_idx,
            batch_size=batch_size,
            flow_dt=flow_dt,
        )
        kv_cache.cache_k[ode_idx, :, :, :, dst, :].copy_(new_k)
        kv_cache.cache_v[ode_idx, :, :, :, dst, :].copy_(new_v)
    kv_cache.valid_tokens = prefix_len
    return z


# --------------------------------------------------------------------------- #
# CUDA graph for the Qwen single-token decode step
# --------------------------------------------------------------------------- #


def _static_cache_ptr(cache) -> int:
    layers = getattr(cache, "layers", None)
    if layers:
        return layers[0].keys.data_ptr()
    key_cache = getattr(cache, "key_cache", None)
    if key_cache:
        return key_cache[0].data_ptr()
    return id(cache)


def _get_static_token_step(self, *, signature, compile_step):
    step = _ORIG["_get_static_token_step"](self, signature=signature, compile_step=compile_step)
    if (
        not _SETTINGS["cuda_graphs"]
        or _SETTINGS["compile_llm"]
        or getattr(self, "_t4_graph_failed", False)
    ):
        return step

    def graphed(inputs_embeds, cache, cache_position):
        if not inputs_embeds.is_cuda:
            return step(inputs_embeds, cache, cache_position)
        graphs = self.__dict__.setdefault("_t4_llm_graphs", {})
        key = (
            inputs_embeds.data_ptr(),
            cache_position.data_ptr(),
            id(cache),
            _static_cache_ptr(cache),
        )
        entry = graphs.get(key)
        if entry is None:
            try:
                # The warm-up writes this token's K/V at this position; the replay
                # below writes the same values again, so it is harmless.
                side = torch.cuda.Stream()
                side.wait_stream(torch.cuda.current_stream())
                with torch.cuda.stream(side):
                    step(inputs_embeds, cache, cache_position)
                torch.cuda.current_stream().wait_stream(side)
                entry = _capture_graph(lambda: step(inputs_embeds, cache, cache_position))
                graphs[key] = entry
            except Exception as exc:
                self._t4_graph_failed = True
                graphs.clear()
                print(f"⚠️ LLM CUDA graph capture failed, using eager decode: {exc}")
                return step(inputs_embeds, cache, cache_position)
        entry[0].replay()
        return entry[1].clone()

    return graphed


def _prefill_exact_length(self, **_kwargs) -> bool:
    # Without torch.compile the 192-patch padded prefill only wastes compute.
    return False


def _patch_solver() -> None:
    from dots_tts.modules.backbone.llm_inference import LLMInference

    _ORIG.setdefault("_ensure_cache", di.DiTSolver._ensure_cache)
    _ORIG.setdefault("_decode_with_kv_cache", di.DiTSolver._decode_with_kv_cache)
    _ORIG.setdefault("_prefill_compiled", di.CachedDiTRunner._prefill_compiled)
    _ORIG.setdefault("_get_static_token_step", LLMInference._get_static_token_step)
    di.DiTSolver._ensure_cache = _ensure_cache
    di.DiTSolver._decode_with_kv_cache = _decode_with_kv_cache
    di.CachedDiTRunner._prefill_compiled = _prefill_exact_length
    LLMInference._get_static_token_step = _get_static_token_step


# --------------------------------------------------------------------------- #
# Low-RAM loading
# --------------------------------------------------------------------------- #


def _from_pretrained_on_gpu(cls, pretrained_model_name_or_path):
    if not torch.cuda.is_available():
        return _ORIG["from_pretrained"](cls, pretrained_model_name_or_path)
    try:
        with torch.device("cuda"):
            return _ORIG["from_pretrained"](cls, pretrained_model_name_or_path)
    except Exception as exc:
        print(f"⚠️ Building the model on the GPU failed ({exc}); building it on the CPU instead.")
        gc.collect()
        torch.cuda.empty_cache()
        return _ORIG["from_pretrained"](cls, pretrained_model_name_or_path)


def _copy_into(target: torch.Tensor, source: torch.Tensor, key: str, path) -> None:
    if tuple(target.shape) != tuple(source.shape):
        raise RuntimeError(
            f"Failed to load {path}: {key} has shape {tuple(source.shape)}, "
            f"model expects {tuple(target.shape)}."
        )
    with torch.no_grad():
        target.copy_(source)


def _load_artifact_module_streaming(cls, module, path):
    """Same result as upstream's load_file + load_state_dict, one tensor at a time."""
    from safetensors import safe_open

    targets = module.state_dict(keep_vars=True)
    total_bytes = max(1, os.path.getsize(path))
    report = _SETTINGS["load_progress"]
    done_bytes = 0
    loaded: set[str] = set()
    unexpected: list[str] = []
    with safe_open(str(path), framework="pt", device="cpu") as handle:
        for key in handle.keys():
            if key not in targets:
                unexpected.append(key)
                continue
            tensor = handle.get_tensor(key)
            _copy_into(targets[key], tensor, key, path)
            done_bytes += tensor.numel() * tensor.element_size()
            del tensor
            loaded.add(key)
            if report is not None:
                report(os.path.basename(str(path)), done_bytes, total_bytes)
        for redundant_key, canonical_key in cls._ARTIFACT_ALIASES:
            if redundant_key in targets and redundant_key not in loaded and canonical_key in loaded:
                _copy_into(targets[redundant_key], handle.get_tensor(canonical_key), redundant_key, path)
                loaded.add(redundant_key)
    missing = [key for key in targets if key not in loaded]
    if missing or unexpected:
        raise RuntimeError(
            f"Failed to load {path}: missing_keys={missing[:10]} unexpected_keys={unexpected[:10]}"
        )
    return module


def _patch_loading() -> None:
    from dots_tts.models.dots_tts.model import DotsTtsModel

    _ORIG.setdefault("from_pretrained", DotsTtsModel.__dict__["from_pretrained"].__func__)
    _ORIG.setdefault("_load_artifact_module", DotsTtsModel.__dict__["_load_artifact_module"].__func__)
    DotsTtsModel.from_pretrained = classmethod(_from_pretrained_on_gpu)
    DotsTtsModel._load_artifact_module = classmethod(_load_artifact_module_streaming)


# --------------------------------------------------------------------------- #
# Reporting
# --------------------------------------------------------------------------- #


def format_profile(result: dict[str, Any]) -> str:
    """One line per stage from ``runtime.generate(..., profile_inference=True)``."""
    profiling = result.get("profiling") or {}
    lines = []
    for stage, stats in profiling.items():
        if int(stats.get("count", 0)) > 0:
            lines.append(
                f"{stage:>15}: {stats['seconds']:.2f}s  calls={int(stats['count'])}  "
                f"rtf={stats.get('rtf', float('nan')):.3f}"
            )
    return "\n".join(lines)
'''
with open("dots_tts_t4_accel.py", "w") as f:
    f.write(ACCEL_SOURCE)

# Optimize constraints
url = "https://raw.githubusercontent.com/studio-dots-ai/dots.tts/main/constraints/recommended.txt"
local_constraints = "constraints.txt"
try:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        content = response.read().decode("utf-8")

    lines = content.splitlines()
    filtered_lines = []
    skip_packages = {"torch", "torchaudio", "numpy", "numba", "gradio"}
    for line in lines:
        line_strip = line.strip()
        if not line_strip or line_strip.startswith("#"):
            filtered_lines.append(line)
            continue
        match = re.match(r"^([a-zA-Z0-9_\-]+)", line_strip)
        if match and match.group(1).lower() in skip_packages:
            continue
        filtered_lines.append(line)

    with open(local_constraints, "w") as f:
        f.write("\n".join(filtered_lines))
except Exception:
    local_constraints = None
time.sleep(0.5)

# ── Stage 4: Python Dependencies Installation ──
render_dashboard(4, "Installing dots.tts and core dependencies...")
log_file = "/tmp/pip_install.log"
with open(log_file, "w") as f:
    f.write("=== Dots.TTS Installation Log ===\n")

pip_cmd_dots = [sys.executable, "-m", "pip", "install", "--no-warn-conflicts"]
if local_constraints and os.path.exists(local_constraints):
    pip_cmd_dots += ["-c", local_constraints]
pip_cmd_dots += [f"./{REPO_DIR}"]
run_quiet_process(pip_cmd_dots, 4, log_file)

pip_cmd_aux = [sys.executable, "-m", "pip", "install", "--no-warn-conflicts", "gradio>=5.0.0", "soundfile>=0.13.1", "transformers>=4.57.0", "librosa>=0.11.0"]
run_quiet_process(pip_cmd_aux, 4, log_file)

# ── Stage 5: Runtime & Module Imports Verification ──
render_dashboard(5, "Validating imports and tensor precision...")
for p in [f"/content/{REPO_DIR}", os.path.abspath(REPO_DIR)]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import soundfile as sf
import gradio as gr
from transformers import pipeline
from dots_tts.runtime import DotsTtsRuntime

env_summary = {
    "NumPy Version": f"{np.__version__} (Pre-installed preserved)",
    "Gradio Version": f"{gr.__version__}",
    "PyTorch Version": f"{torch.__version__} (CUDA active)",
    "DotsTtsRuntime": "Ready (T4 CUDA-graph engine: dots_tts_t4_accel.py)"
}

render_dashboard(TOTAL_STAGES + 1, env_summary=env_summary)

In [ ]:
#@title 🚀 Step 2: Launch Dots.TTS Voice Cloning Studio
# Launches the Gradio Studio (Cloudflare tunnel + Gradio share links). The selected model is downloaded and loaded when you click Generate, with one clean progress bar per request here and in the UI.
import gc
import hashlib
import logging
import os
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import traceback
import warnings

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TORCH_LOGS"] = "-all"

for p in ["/content/dots.tts", os.path.abspath("dots.tts"), os.getcwd()]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import gradio as gr
import librosa
import numpy as np
import scipy.signal
import soundfile as sf
import torch
import transformers
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.constants import HF_HUB_CACHE
from huggingface_hub.utils import disable_progress_bars
from loguru import logger as loguru_logger
from transformers import pipeline
from dots_tts.models.dots_tts.model import DotsTtsModel
from dots_tts.runtime import DotsTtsRuntime

import dots_tts_t4_accel as accel

# ── Quiet logs: only our progress bar and real errors reach this cell ──
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="ERROR", format="❌ {message}")
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
disable_progress_bars()
for noisy in ("httpx", "gradio", "urllib3", "huggingface_hub", "asyncio"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_LABEL = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "No GPU"

MODELS = {
    "dots-studio/dots.tts-mf": {"label": "⚡ MeanFlow · 4 steps (Fastest)", "steps": 4, "cfg": False},
    "dots-studio/dots.tts-soar": {"label": "🎯 SOAR · 10 steps (Best voice match)", "steps": 10, "cfg": True},
    "dots-studio/dots.tts-base": {"label": "🧱 Base · 10 steps (Foundation)", "steps": 10, "cfg": True},
}
DEFAULT_MODEL = "dots-studio/dots.tts-mf"
# The model's longest single take is 512 patches of 160 ms (81.9 s) including the
# reference audio, so long text is generated in parts of about a minute each.
MAX_PATCHES = 512
PART_SECONDS = 60.0
PATCH_SECONDS = 0.16
MAX_REF_SECONDS = 15.0
CHARS_PER_SECOND = 14.0  # speaking-rate guess for part sizes and the progress bar (CJK chars count x3)
ASR_MODEL = "openai/whisper-large-v3-turbo"
# "Match reference pace": only slow sections that run faster than the reference by
# more than PACE_THRESHOLD, correct PACE_DAMPING of the excess, never more than MAX.
PACE_THRESHOLD = 1.05
PACE_DAMPING = 0.75
MAX_PACE_SLOWDOWN = 1.25

# ── One progress bar per request: a bar in the web page (polled, so it also works
# through the Cloudflare tunnel) + a single updating line in this cell ──
PROGRESS_STATE = {"active": False, "fraction": 0.0, "desc": "", "title": "", "summary": ""}

class RunProgress:
    WIDTH = 30

    def __init__(self, title):
        self.fraction = 0.0
        self.last_draw = 0.0
        PROGRESS_STATE.update(active=True, fraction=0.0, desc="Starting...", title=title, summary="")
        print(f"\n🎙️ {title}", flush=True)

    def update(self, fraction, desc, force=False):
        self.fraction = max(self.fraction, min(float(fraction), 1.0))
        PROGRESS_STATE.update(fraction=self.fraction, desc=desc)
        now = time.time()
        if not force and now - self.last_draw < 0.25:
            return
        self.last_draw = now
        filled = int(self.WIDTH * self.fraction)
        bar = "█" * filled + "░" * (self.WIDTH - filled)
        print(f"\r   [{bar}] {self.fraction * 100:5.1f}%  {desc}".ljust(100), end="", flush=True)

    def finish(self, summary):
        self.update(1.0, "Done", force=True)
        PROGRESS_STATE.update(active=False, summary=summary)
        print(f"\n   {summary}", flush=True)

    def fail(self, message):
        PROGRESS_STATE.update(active=False, summary=f"❌ {message}")
        print(f"\n   ❌ {message}", flush=True)

def progress_html():
    state = dict(PROGRESS_STATE)
    if state["active"]:
        pct = state["fraction"] * 100
        return (f'<div class="run-progress"><div class="run-progress-head"><span>{state["desc"]}</span>'
                f'<span>{pct:.0f}%</span></div><div class="run-progress-track">'
                f'<div class="run-progress-fill" style="width:{pct:.1f}%"></div></div></div>')
    if state["summary"]:
        return f'<div class="run-progress run-progress-idle">{state["summary"]}</div>'
    return '<div class="run-progress run-progress-idle">Ready. Pick a model and click Generate Audio.</div>'

# Called by the model after every generated 160 ms patch
PATCH_LISTENER = [None]
if not hasattr(DotsTtsModel, "_aiq_consume_audio_patch"):
    DotsTtsModel._aiq_consume_audio_patch = DotsTtsModel._consume_audio_patch

def _consume_audio_patch_with_progress(self, state, **kwargs):
    DotsTtsModel._aiq_consume_audio_patch(self, state, **kwargs)
    if PATCH_LISTENER[0] is not None:
        PATCH_LISTENER[0]()

DotsTtsModel._consume_audio_patch = _consume_audio_patch_with_progress

# ── Whisper auto-transcription (GPU fp16, cached per reference file) ──
ASR_PIPELINE = None
TRANSCRIPTS = {}

def reference_key(path):
    with open(path, "rb") as f:
        return hashlib.sha1(f.read()).hexdigest()

def transcribe(path, key):
    """Accurate transcript of the reference. The model reads the speaker's pace from
    transcript vs audio, so missing or invented words change the output's speed.
    Whisper sits on the GPU only while it transcribes."""
    global ASR_PIPELINE
    if key not in TRANSCRIPTS:
        if ASR_PIPELINE is None:
            ASR_PIPELINE = pipeline(
                "automatic-speech-recognition",
                model=ASR_MODEL,
                device=0 if DEVICE == "cuda" else -1,
                torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
                chunk_length_s=30,
            )
        try:
            ASR_PIPELINE.model.to(DEVICE)
            TRANSCRIPTS[key] = ASR_PIPELINE(path)["text"].strip()
        finally:
            ASR_PIPELINE.model.to("cpu")
            torch.cuda.empty_cache()
    return TRANSCRIPTS[key]

def _pause_cut(audio, sr, max_seconds, min_seconds=6.0):
    """Sample index of the last clear pause before max_seconds, so no word is cut in half.

    Prefers a sentence-like pause (>= 0.3 s), then any short pause (>= 0.12 s).
    Returns None when the speech never pauses in that window.
    """
    intervals = librosa.effects.split(audio, top_db=35, frame_length=1024, hop_length=256)
    lo, hi = int(min_seconds * sr), int(max_seconds * sr)
    pauses = []
    for (_, end), (next_start, _) in zip(intervals[:-1], intervals[1:]):
        gap = next_start - end
        if lo <= end and end + min(gap, int(0.15 * sr)) <= hi:
            pauses.append((end, gap))
    for min_gap in (0.3, 0.12):
        candidates = [(end, gap) for end, gap in pauses if gap >= min_gap * sr]
        if candidates:
            end, gap = candidates[-1]
            return end + min(gap // 2, int(0.15 * sr))
    return None

def prepare_reference(path):
    """The reference shares each take's length budget, so keep at most MAX_REF_SECONDS.

    Long references are cut at the last pause before the limit: a cut mid-word makes
    the transcript describe more speech than the clip holds, and the model then thinks
    the speaker talks faster than they do.
    """
    seconds = librosa.get_duration(path=path)
    if seconds <= MAX_REF_SECONDS:
        return path, False, seconds
    audio, sr = librosa.load(path, sr=None, mono=True)
    cut = _pause_cut(audio, sr, MAX_REF_SECONDS) or int(MAX_REF_SECONDS * sr)
    audio = audio[:cut].copy()
    fade = min(len(audio), int(0.02 * sr))
    audio[-fade:] *= np.linspace(1.0, 0.0, fade, dtype=audio.dtype)
    out_path = os.path.join(tempfile.mkdtemp(), "reference_trimmed.wav")
    sf.write(out_path, audio, sr)
    return out_path, True, len(audio) / sr

# ── Speaking pace ──
def _spoken_units(text):
    return sum(1 for ch in text if ch.isalnum())

def _cjk_share(text):
    units = _spoken_units(text)
    return sum(1 for ch in text if "\u3000" <= ch <= "\u9fff") / units if units else 0.0

def speaking_rate(audio, sample_rate, text):
    """Spoken characters per second of voiced audio (pauses excluded), or None."""
    intervals = librosa.effects.split(audio, top_db=35, frame_length=1024, hop_length=256)
    voiced = sum(end - start for start, end in intervals) / sample_rate
    units = _spoken_units(text)
    return units / voiced if voiced > 0.5 and units > 0 else None

def _syllable_peaks(segment, sample_rate):
    """Syllable count from loudness peaks (the de Jong & Wempe method, simplified)."""
    hop = int(0.01 * sample_rate)
    rms = librosa.feature.rms(y=segment, frame_length=int(0.03 * sample_rate), hop_length=hop)[0]
    window = np.hanning(9)
    rms = np.convolve(rms, window / window.sum(), mode="same")
    db = librosa.amplitude_to_db(rms + 1e-8, ref=np.max)
    peaks, _ = scipy.signal.find_peaks(db, prominence=2.0, distance=8, height=-35)
    return len(peaks)

def _pace_sections(audio, sample_rate, min_voiced_seconds=3.0):
    """(start, end, voiced_samples) sections of about 3 s of speech, cut in pauses."""
    intervals = librosa.effects.split(audio, top_db=35, frame_length=1024, hop_length=256)
    groups, current, voiced = [], [], 0
    for start, end in intervals:
        current.append((start, end))
        voiced += end - start
        if voiced >= min_voiced_seconds * sample_rate:
            groups.append(current)
            current, voiced = [], 0
    if current:
        if groups and voiced < min_voiced_seconds * sample_rate / 2:
            groups[-1] += current
        else:
            groups.append(current)
    bounds = [0] + [(a[-1][1] + b[0][0]) // 2 for a, b in zip(groups[:-1], groups[1:])] + [len(audio)]
    return [(bounds[i], bounds[i + 1], sum(e - s for s, e in group)) for i, group in enumerate(groups)]

def match_reference_pace(audio, sample_rate, text, ref_rate):
    """Slow only the sections that run faster than the reference speaker.

    Pace is measured per ~3 s section and fitted with a straight line, because the
    model tends to speed up gradually toward the end of a long take. The line is
    calibrated to the part's text, so its average equals the part's real rate.
    Returns (audio, largest_slowdown).
    """
    sections = _pace_sections(audio, sample_rate)
    units = _spoken_units(text)
    if not sections or not ref_rate or units == 0:
        return audio, 1.0
    peaks = np.array([max(1, _syllable_peaks(audio[s:e], sample_rate)) for s, e, _ in sections], dtype=float)
    voiced = np.array([max(v, 1) / sample_rate for _, _, v in sections])
    centers = np.array([(s + e) / 2 / sample_rate for s, e, _ in sections])
    local = peaks / voiced
    if len(sections) >= 3:
        slope, intercept = np.polyfit(centers, local, 1, w=voiced)
        local = np.maximum(intercept + slope * centers, 0.1)
    chars_per_second = local * units / peaks.sum()
    ratios = chars_per_second / ref_rate
    stretches = np.where(ratios > PACE_THRESHOLD,
                         np.clip(1.0 + PACE_DAMPING * (ratios - 1.0), 1.0, MAX_PACE_SLOWDOWN), 1.0)
    if stretches.max() <= 1.0:
        return audio, 1.0
    out = [time_stretch(audio[s:e], sample_rate, r) for (s, e, _), r in zip(sections, stretches)]
    return np.concatenate(out).astype(np.float32), float(stretches.max())

def time_stretch(audio, sample_rate, duration_ratio):
    """Change duration without changing pitch (>1 = slower). Uses Rubber Band, which
    keeps speech clean; falls back to librosa if it is missing."""
    if abs(duration_ratio - 1.0) < 0.01:
        return audio
    if shutil.which("rubberband"):
        work = tempfile.mkdtemp()
        src, dst = os.path.join(work, "in.wav"), os.path.join(work, "out.wav")
        sf.write(src, audio, sample_rate, subtype="FLOAT")
        for engine in (["--fine"], []):  # --fine is Rubber Band 3's best engine
            done = subprocess.run(["rubberband", "-q", *engine, "-t", f"{duration_ratio:.4f}", src, dst],
                                  capture_output=True)
            if done.returncode == 0 and os.path.isfile(dst):
                out, _ = sf.read(dst, dtype="float32")
                return out if out.ndim == 1 else out.mean(axis=1)
    return librosa.effects.time_stretch(audio, rate=1.0 / duration_ratio)

# ── Model download + load (only when you click Generate; one model in VRAM at a time) ──
CURRENT_MODEL_PATH = None
RUNTIME = None
WEIGHT_FILES = ("model.safetensors", "vocoder.safetensors", "speaker_encoder.safetensors")

def download_model(repo_id, report):
    """Snapshot download with a real percentage (Hub progress bars are off)."""
    try:
        cached = snapshot_download(repo_id, local_files_only=True)
        if all(os.path.isfile(os.path.join(cached, name)) for name in WEIGHT_FILES):
            return cached
    except Exception:
        pass
    try:
        info = HfApi().model_info(repo_id, files_metadata=True)
        total = sum((f.size or 0) for f in info.siblings)
    except Exception:
        total = 0
    result = {}

    def worker():
        try:
            result["path"] = snapshot_download(repo_id)
        except Exception as exc:
            result["error"] = exc

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    blobs = os.path.join(HF_HUB_CACHE, "models--" + repo_id.replace("/", "--"), "blobs")
    while thread.is_alive():
        if total and os.path.isdir(blobs):
            done = sum(entry.stat().st_size for entry in os.scandir(blobs) if entry.is_file())
            report(min(done / total, 1.0), f"Downloading model · {done / 1e9:.2f} / {total / 1e9:.2f} GB")
        thread.join(0.5)
    if "error" in result:
        raise result["error"]
    return result["path"]

def get_runtime(model_path, run, start, end):
    """Returns the runtime for model_path, loading it if needed. Progress goes from start to end."""
    global CURRENT_MODEL_PATH, RUNTIME
    if RUNTIME is not None and CURRENT_MODEL_PATH == model_path:
        return RUNTIME
    if RUNTIME is not None:
        run.update(start, "Unloading previous model...", force=True)
        RUNTIME = None
        CURRENT_MODEL_PATH = None
        gc.collect()
        torch.cuda.empty_cache()

    cfg = MODELS[model_path]
    span = end - start
    local_path = download_model(model_path, lambda f, desc: run.update(start + 0.6 * span * f, desc))

    total_bytes = sum(os.path.getsize(os.path.join(local_path, name)) for name in WEIGHT_FILES)
    loaded = {}

    def on_weights(name, done, _total):
        loaded[name] = done
        f = min(sum(loaded.values()) / total_bytes, 1.0)
        run.update(start + span * (0.6 + 0.38 * f), f"Loading model to GPU · {f * 100:.0f}%")

    accel.apply(max_bucket=MAX_PATCHES)
    accel.set_load_progress(on_weights)
    run.update(start + 0.6 * span, "Loading model to GPU...", force=True)
    try:
        RUNTIME = DotsTtsRuntime.from_pretrained(
            local_path,
            precision="float16",
            optimize=True,
            warmup_on_optimize=False,
            max_generate_length=MAX_PATCHES,
        )
    finally:
        accel.set_load_progress(None)
    CURRENT_MODEL_PATH = model_path
    return RUNTIME

def _weight(text):
    return sum(3 if "\u3000" <= ch <= "\u9fff" else 1 for ch in text)

def _split_long(piece, max_units):
    """Split an over-long sentence at commas, then at words (or characters for CJK)."""
    if _weight(piece) <= max_units:
        return [piece]
    out = []
    for clause in filter(None, re.split(r"(?<=[,，、:：])\s*", piece)):
        if _weight(clause) <= max_units:
            out.append(clause)
            continue
        sep = " " if " " in clause else ""
        current = ""
        for word in (clause.split(" ") if sep else list(clause)):
            candidate = f"{current}{sep}{word}" if current else word
            if current and _weight(candidate) > max_units:
                out.append(current)
                current = word
            else:
                current = candidate
        if current:
            out.append(current)
    return out

def split_text(text, max_units):
    """Sentence-aligned parts of similar size, none longer than max_units."""
    sentences = filter(None, re.split(r"(?<=[.!?;。！？；])\s*", text.strip()))
    pieces = [p for sentence in sentences for p in _split_long(sentence, max_units)]
    count = max(1, int(np.ceil(_weight(text) / max_units)))
    target = _weight(text) / count
    chunks, current = [], ""
    for piece in pieces:
        if current and (_weight(current) + _weight(piece) > max_units or _weight(current) >= target):
            chunks.append(current)
            current = piece
        else:
            sep = "" if not current or _weight(piece[:1]) == 3 else " "
            current = f"{current}{sep}{piece}"
    return chunks + ([current] if current else [])

def join_parts(parts, sample_rate):
    """Trim each part's edge silence, then join with a natural pause and short fades."""
    if len(parts) == 1:
        return parts[0]
    pause = np.zeros(int(0.25 * sample_rate), dtype=np.float32)
    fade = int(0.01 * sample_rate)
    ramp = np.linspace(0.0, 1.0, fade, dtype=np.float32)
    joined = []
    for part in parts:
        part, _ = librosa.effects.trim(part, top_db=45)
        part = part.copy()
        if len(part) > 2 * fade:
            part[:fade] *= ramp
            part[-fade:] *= ramp[::-1]
        joined += [part, pause]
    return np.concatenate(joined[:-1]).astype(np.float32)

# ── Inference handler ──
REQUEST_COUNT = [0]

def synthesize(ref_audio, ref_text, gen_text, model_name, num_steps, guidance_scale, speaker_scale, speed_factor, match_pace=False):
    if DEVICE == "cpu":
        return None, "❌ Error: CPU runtime detected! Go to Runtime -> Change runtime type and select T4 GPU."
    if ref_audio is None:
        return None, "⚠️ Please upload or record a Reference Audio first."
    if not gen_text or not gen_text.strip():
        return None, "⚠️ Please enter the text you want to synthesize."

    cfg = MODELS[model_name]
    REQUEST_COUNT[0] += 1
    run = RunProgress(f"Request #{REQUEST_COUNT[0]} · {cfg['label']}")
    t_start = time.time()
    try:
        ref_path, trimmed, ref_seconds = prepare_reference(ref_audio)
        text = gen_text.strip()
        if text[-1] not in ".!?,;。！？，；":
            text += "."

        # Each part is one take: the reference and that part's speech must fit together.
        available_patches = MAX_PATCHES - int(np.ceil(ref_seconds / PATCH_SECONDS)) - 2
        available_seconds = available_patches * PATCH_SECONDS
        part_seconds = min(PART_SECONDS, 0.85 * available_seconds)  # headroom for slow speakers
        chunks = split_text(text, part_seconds * CHARS_PER_SECOND)

        ref_text_clean = "" if trimmed else (ref_text or "").strip()
        ref_key = reference_key(ref_path)
        needs_asr = not ref_text_clean and ref_key not in TRANSCRIPTS
        needs_load = CURRENT_MODEL_PATH != model_name

        load_end = 0.35 if needs_load else 0.0
        asr_end = load_end + (0.05 if needs_asr else 0.0)
        synth_end = 0.97

        run.update(0.0, "Preparing...", force=True)
        runtime = get_runtime(model_name, run, 0.0, load_end)

        if not ref_text_clean:
            run.update(load_end, "Transcribing reference audio (Whisper)...", force=True)
            ref_text_clean = transcribe(ref_path, ref_key)

        ref_wave, ref_sr = librosa.load(ref_path, sr=None, mono=True)
        ref_wave, _ = librosa.effects.trim(ref_wave, top_db=30)
        ref_rate = speaking_rate(ref_wave, ref_sr, ref_text_clean)
        same_script = abs(_cjk_share(ref_text_clean) - _cjk_share(text)) < 0.5
        pace_fixes = []

        sampling = {"num_steps": int(num_steps)}
        if cfg["cfg"]:
            sampling["guidance_scale"] = float(guidance_scale)

        estimates = [max(8.0, _weight(c) / CHARS_PER_SECOND / PATCH_SECONDS) for c in chunks]
        total_estimate = sum(estimates)
        pieces, gen_seconds, sample_rate, hit_limit = [], 0.0, 48000, False
        done_estimate = 0.0
        for i, chunk in enumerate(chunks):
            part = f" · part {i + 1}/{len(chunks)}" if len(chunks) > 1 else ""
            patches = [0]

            def on_patch(i=i, part=part):
                patches[0] += 1
                f = (done_estimate + min(patches[0], estimates[i] * 0.98)) / total_estimate
                run.update(asr_end + (synth_end - asr_end) * f,
                           f"Generating speech{part} · {patches[0] * PATCH_SECONDS:.1f}s of audio")

            run.update(asr_end + (synth_end - asr_end) * done_estimate / total_estimate,
                       f"Generating speech{part}...", force=True)
            PATCH_LISTENER[0] = on_patch
            try:
                with torch.inference_mode():
                    result = runtime.generate(
                        text=chunk,
                        prompt_audio_path=ref_path,
                        prompt_text=ref_text_clean,
                        speaker_scale=float(speaker_scale),
                        **sampling,
                    )
            finally:
                PATCH_LISTENER[0] = None
            done_estimate += estimates[i]
            hit_limit = hit_limit or patches[0] >= available_patches
            piece = result["audio"].float().cpu().squeeze().numpy().astype(np.float32)
            gen_seconds += float(result["time_used"])
            sample_rate = int(result["sample_rate"])
            # Keep the reference speaker's pace where this part runs faster.
            if match_pace and ref_rate and same_script:
                run.update(run.fraction, f"Matching reference pace{part}...", force=True)
                piece, slowdown = match_reference_pace(piece, sample_rate, chunk, ref_rate)
                if slowdown > 1.0:
                    pace_fixes.append(slowdown)
            pieces.append(piece)

        if len(pieces) > 1:
            run.update(synth_end, "Joining parts...", force=True)
        audio = join_parts(pieces, sample_rate)
        duration = len(audio) / sample_rate

        if abs(float(speed_factor) - 1.0) > 0.01:
            run.update(synth_end, "Adjusting speech speed...", force=True)
            audio = time_stretch(audio, sample_rate, 1.0 / float(speed_factor))

        run.update(0.99, "Saving WAV...", force=True)
        out_path = os.path.join(tempfile.mkdtemp(), "output.wav")
        sf.write(out_path, audio, sample_rate)

        rtf = gen_seconds / max(duration, 1e-6)
        run.finish(f"✅ {duration:.1f}s of audio · generation {gen_seconds:.1f}s (RTF {rtf:.2f}) · total {time.time() - t_start:.1f}s")
        status_msg = (
            f"✅ Done with {cfg['label']}\n"
            f"⏱️ Generation: {gen_seconds:.2f}s for {duration:.2f}s of audio | RTF: {rtf:.3f} | Parts: {len(chunks)}\n"
            f"📝 Reference Transcript: {ref_text_clean}"
        )
        if pace_fixes:
            status_msg += f"\n🎚️ Matched the reference pace: slowed the faster sections of {len(pace_fixes)}/{len(chunks)} part(s), by up to {max(pace_fixes):.2f}x"
        if hit_limit:
            status_msg += "\n⚠️ A part reached the model's maximum take length, so its end may be cut off. A shorter reference gives each part more room."
        if trimmed:
            status_msg += f"\nℹ️ Reference was longer than {MAX_REF_SECONDS:.0f}s: used the first {ref_seconds:.1f}s (cut at a pause) and auto-transcribed it."
        return out_path, status_msg

    except Exception as e:
        run.fail(f"{type(e).__name__}: {e}")
        traceback.print_exc()
        return None, f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"

# ── Cloudflare Tunnel Helper ──
def start_cloudflare_tunnel(port=7860):
    cf_bin = "/usr/local/bin/cloudflared"
    if not os.path.exists(cf_bin):
        cf_bin = shutil.which("cloudflared")
    if not cf_bin:
        try:
            print("🌐 Installing cloudflared binary...")
            os.system("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
            cf_bin = "/usr/local/bin/cloudflared"
        except Exception:
            return None, None

    log_file = "/tmp/cloudflared.log"
    try:
        if os.path.exists(log_file):
            os.remove(log_file)
        log_f = open(log_file, "w")
        proc = subprocess.Popen(
            [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=log_f,
            stderr=subprocess.STDOUT
        )
    except Exception:
        return None, None

    tunnel_url = None
    for _ in range(30):
        time.sleep(0.5)
        if os.path.exists(log_file):
            try:
                with open(log_file, "r") as f:
                    matches = re.findall(r"https://[a-zA-Z0-9.-]+[.]trycloudflare[.]com", f.read())
                    if matches:
                        tunnel_url = matches[0]
                        break
            except Exception:
                pass

    return proc, tunnel_url

# ── Gradio Styling (AIQUEST standard) ──
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
.run-progress { margin: 10px 0 4px 0; font-size: 13px; color: #374151; }
.run-progress-head { display: flex; justify-content: space-between; margin-bottom: 6px; font-weight: 600; }
.run-progress-track { height: 10px; border-radius: 999px; background: #e5e7eb; overflow: hidden; }
.run-progress-fill { height: 100%; border-radius: 999px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); transition: width 0.4s ease; }
.run-progress-idle { color: #6b7280; font-weight: 500; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 14px 0; }
.footer { text-align: center; padding: 22px; margin-top: 32px; border-top: 1px solid #e5e7eb; color: #6b7280; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">⚡ Dots.TTS - Zero-Shot Voice Cloning</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Google Colab T4 Edition · CUDA-Graph Engine</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

STATUS_HTML = f'<div class="status-line">⚡ T4 accelerator active: CUDA-graph DiT + Qwen decode · float16 · {GPU_LABEL}</div>'

with gr.Blocks(title="Dots.TTS - Zero-Shot Voice Cloning | AIQUEST") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)

    gr.Markdown(
        "Upload a reference voice sample (3 to 15 seconds), type the target text, "
        "and hit **Generate Audio**. If the reference transcript is blank, Whisper transcribes it automatically. "
        "**MeanFlow 4-step** is the fastest; **SOAR** gives the closest voice match. "
        "The selected model downloads and loads on the first click (switching models reloads)."
    )

    with gr.Row():
        with gr.Column(scale=1):
            ref_audio = gr.Audio(label="📎 Reference Audio (3-15s)", type="filepath")
            ref_text = gr.Textbox(label="Reference Transcript (Optional)", placeholder="Leave blank to auto-transcribe...")
            gen_text = gr.Textbox(label="Text to Synthesize (long text is generated in ~1-minute parts)", placeholder="Enter target text to generate speech...", lines=4)
            model_name = gr.Dropdown(
                label="Model",
                choices=[(cfg["label"], path) for path, cfg in MODELS.items()],
                value=DEFAULT_MODEL,
            )

        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Advanced Inference Parameters", open=False):
                num_steps = gr.Slider(label="⚡ Sampling Steps", minimum=2, maximum=32, step=1, value=MODELS[DEFAULT_MODEL]["steps"])
                guidance_scale = gr.Slider(label="🎯 Classifier Free Guidance (CFG)", minimum=0.5, maximum=2.0, step=0.1, value=1.2, interactive=MODELS[DEFAULT_MODEL]["cfg"])
                speaker_scale = gr.Slider(label="👤 Speaker Intensity", minimum=0.5, maximum=2.5, step=0.1, value=1.5)
                match_pace = gr.Checkbox(label="🎯 Match reference pace", value=False, info="Gently slows the sections (usually near the end) that come out faster than the reference speaker")
                speed_factor = gr.Slider(label="🎚️ Speech Speed (Time Stretch)", minimum=0.5, maximum=2.0, step=0.05, value=1.0)

            with gr.Row():
                gen_btn = gr.Button("🎬 Generate Audio", variant="primary", size="lg", elem_id="gen-btn")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

            progress_bar = gr.HTML(progress_html())
            audio_out = gr.Audio(label="🔊 Synthesized Audio Output", type="filepath")
            status_out = gr.Textbox(label="Status & Metrics Output", lines=4, interactive=False)

    # MeanFlow is distilled for 4 steps with guidance fused in; SOAR/Base use 10 steps + CFG.
    def update_model_params(model):
        cfg = MODELS[model]
        return gr.update(value=cfg["steps"]), gr.update(value=1.2, interactive=cfg["cfg"])

    model_name.change(fn=update_model_params, inputs=[model_name], outputs=[num_steps, guidance_scale])

    gen_event = gen_btn.click(
        fn=synthesize,
        inputs=[ref_audio, ref_text, gen_text, model_name, num_steps, guidance_scale, speaker_scale, speed_factor, match_pace],
        outputs=[audio_out, status_out],
        show_progress="minimal",
    )

    # Plain polling requests, so the bar also moves through the Cloudflare tunnel
    progress_timer = gr.Timer(0.5)
    progress_timer.tick(fn=progress_html, outputs=[progress_bar], queue=False, show_progress="hidden")

    def stop_request():
        PROGRESS_STATE.update(active=False, summary="🛑 Stopped.")

    stop_btn.click(fn=stop_request, cancels=[gen_event], queue=False)

    clear_btn.click(
        fn=lambda: (None, "", "", DEFAULT_MODEL, MODELS[DEFAULT_MODEL]["steps"], gr.update(value=1.2, interactive=MODELS[DEFAULT_MODEL]["cfg"]), 1.5, 1.0, False, None, ""),
        outputs=[ref_audio, ref_text, gen_text, model_name, num_steps, guidance_scale, speaker_scale, speed_factor, match_pace, audio_out, status_out]
    )

    gr.HTML('<div class="footer"><p style="font-size: 15px; margin: 4px 0;">🎙️ Created by <strong>AIQUEST Academy</strong></p><p style="font-size: 13px; margin: 4px 0; color: #9ca3af;">Free &amp; Open Source - Zero-Shot Voice Cloning - Google Colab T4 GPU Edition</p><p style="font-size: 13px; margin: 8px 0;"><a href="https://youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a> | <a href="https://aiquest.site" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">aiquest.site</a></p></div>')

SERVER_PORT = 7860
print("\n" + "=" * 65)
print("🚀 Starting Dots.TTS Studio & Cloudflare Tunnel...")
print("=" * 65)

# 1. Start Cloudflare Tunnel in background
cf_proc, cf_url = start_cloudflare_tunnel(port=SERVER_PORT)

# 2. Launch Gradio Studio (the model loads when you click Generate)
demo.queue(max_size=20, default_concurrency_limit=1)
launch_res = demo.launch(
    share=True,
    inline=False,
    debug=False,
    show_error=True,
    server_port=SERVER_PORT,
    prevent_thread_lock=True,
    quiet=True,
    css=CSS,
    theme=gr.themes.Soft()
)

share_url = getattr(demo, "share_url", None)
if not share_url and isinstance(launch_res, tuple) and len(launch_res) >= 3:
    share_url = launch_res[2]

# If Cloudflare tunnel needed extra seconds, poll once more
if not cf_url and os.path.exists("/tmp/cloudflared.log"):
    try:
        with open("/tmp/cloudflared.log", "r") as f:
            m = re.findall(r"https://[a-zA-Z0-9.-]+[.]trycloudflare[.]com", f.read())
            if m:
                cf_url = m[0]
    except Exception:
        pass

print("\n" + "=" * 65)
print("🎙️ Dots.TTS Studio is LIVE!")
print("=" * 65)
if cf_url:
    print(f"🌐 Cloudflare Tunnel (Recommended): {cf_url}")
else:
    print("🌐 Cloudflare Tunnel:                (Connecting... check /tmp/cloudflared.log)")
if share_url:
    print(f"🔗 Gradio Public Share (Backup):     {share_url}")
print(f"🖥️ Local Instance URL:               http://127.0.0.1:{SERVER_PORT}")
print("⏳ The selected model downloads & loads on your first Generate click.")
print("📊 A progress bar for every request appears below.")
print("=" * 65 + "\n")

# Keep the cell running so request progress keeps streaming here
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Stopped Dots.TTS Studio.")
    if cf_proc:
        cf_proc.terminate()

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---